# Gaps and Islands

## Overview
**Gaps and islands** is a classic SQL pattern for identifying consecutive sequences ("islands") and the breaks between them ("gaps") in ordered data.

**Islands** are groups of consecutive or adjacent rows — accounts active in consecutive months, a sequence of transactions without a flagged break, continuous login sessions.

**Gaps** are the breaks between islands — months with no activity, periods of inactivity, missing sequence numbers.

**The classic island detection technique:**
```sql
row_number - row_number_within_group = constant for consecutive rows
```

More specifically, for rows that should be sequential by integer or date:
```sql
ROW_NUMBER() OVER (ORDER BY seq_col)
- ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY seq_col)
= a constant per island
```

When consecutive rows are subtracted this way, all rows in the same island share the same constant, giving each island a natural group label.

**Session detection** (the time-gap variant) uses LAG to compute time between events, then marks a new session whenever the gap exceeds a threshold.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, account_id INTEGER, trade_date TEXT, ticker TEXT, direction TEXT, shares INTEGER, price REAL, total_value REAL);
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,101,'2023-03-05','Deposit',4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',15000.00,'Payroll',0),(1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',15000.00,'Payroll',0),(1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',15000.00,'Payroll',0),(1012,105,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1014,105,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),(1016,107,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',-25.00,'Fee',1),(1018,107,'2023-03-15','Withdrawal',-450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',2800.00,'Payroll',0),(1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',4500.00,'Payroll',0),(1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',4500.00,'Payroll',0);
INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy',10,148.50,1485.00),(2002,104,'2023-01-25','MSFT','Buy',5,240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy',5,153.20,766.00),(2004,104,'2023-02-28','MSFT','Sell',3,252.80,758.40),
  (2005,104,'2023-03-15','NVDA','Buy',2,278.50,557.00),(2006,108,'2023-01-05','AAPL','Buy',20,130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy',8,95.20,761.60),(2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy',5,99.80,499.00),(2010,108,'2023-03-10','NVDA','Buy',4,265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy',8,238.00,1904.00),(2012,110,'2023-02-01','AAPL','Buy',15,145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy',3,248.50,745.50),(2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy',6,280.00,1680.00);
""")

# Add a sequence-number table to demonstrate integer gaps
conn.executescript("""
CREATE TABLE daily_balances (
    account_id  INTEGER,
    balance_date TEXT,
    balance     REAL
);
INSERT INTO daily_balances VALUES
-- Account 101: active Jan, gap in Feb, active again Mar
  (101,'2023-01-01',4200.00),(101,'2023-01-02',3350.00),(101,'2023-01-03',3230.00),
  (101,'2023-03-01',7200.00),(101,'2023-03-02',6220.00),(101,'2023-03-03',5770.00),
-- Account 103: continuous Jan-Mar
  (103,'2023-01-01',15000.00),(103,'2023-01-02',11800.00),
  (103,'2023-02-01',26800.00),(103,'2023-02-02',24000.00),
  (103,'2023-03-01',39000.00),(103,'2023-03-02',36200.00);

CREATE TABLE login_events (
    event_id    INTEGER PRIMARY KEY,
    account_id  INTEGER,
    event_time  TEXT    -- YYYY-MM-DD HH:MM
);
INSERT INTO login_events VALUES
  (1, 101,'2023-01-05 09:00'),(2,101,'2023-01-05 09:08'),(3,101,'2023-01-05 09:15'),
  (4, 101,'2023-01-05 11:30'),(5,101,'2023-01-05 11:35'),
  (6, 101,'2023-01-05 14:00'),
  (7, 103,'2023-01-05 10:00'),(8,103,'2023-01-05 10:05'),(9,103,'2023-01-05 10:12'),
  (10,103,'2023-01-05 13:00'),(11,103,'2023-01-05 13:04');
""")
conn.commit()
print('Schema ready — including daily_balances and login_events tables.')


---
## Finding gaps — missing dates in a sequence

In [ ]:
# Detect gaps in daily_balances by comparing each date to the previous date
print('=== Gaps in daily balance data per account ===')
q = """
SELECT  account_id,
        balance_date,
        LAG(balance_date) OVER (
            PARTITION BY account_id
            ORDER BY balance_date
        )                                                           AS prev_date,
        CAST(JULIANDAY(balance_date) -
             JULIANDAY(LAG(balance_date) OVER (
                 PARTITION BY account_id ORDER BY balance_date
             )) AS INTEGER)                                         AS days_gap,
        CASE WHEN CAST(JULIANDAY(balance_date) -
                       JULIANDAY(LAG(balance_date) OVER (
                           PARTITION BY account_id ORDER BY balance_date
                       )) AS INTEGER) > 1
             THEN 'GAP DETECTED'
             ELSE 'consecutive'
        END                                                         AS gap_flag
FROM    daily_balances
ORDER BY account_id, balance_date
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Island detection — grouping consecutive rows

In [ ]:
# Classic island technique: row_number over all - row_number within group = island id
print('=== Island detection on transaction types (consecutive same-type runs) ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        txn_type,
        amount,
        -- global row number ordered by date
        ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY txn_date, txn_id) AS rn_all,
        -- row number partitioned by account AND type
        ROW_NUMBER() OVER (PARTITION BY account_id, txn_type
                           ORDER BY txn_date, txn_id)               AS rn_type,
        -- the difference is constant for each island
        ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY txn_date, txn_id)
        - ROW_NUMBER() OVER (PARTITION BY account_id, txn_type
                             ORDER BY txn_date, txn_id)             AS island_key
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date, txn_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Rows sharing the same (account_id, txn_type, island_key) form one consecutive island.')


---
## Counting island size and boundaries

In [ ]:
# Summarise each island: start, end, length
print('=== Island summary: start/end date and length per consecutive run ===')
q = """
WITH numbered AS (
    SELECT  txn_id,
            account_id,
            txn_date,
            txn_type,
            ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY txn_date, txn_id)
            - ROW_NUMBER() OVER (PARTITION BY account_id, txn_type ORDER BY txn_date, txn_id)
                AS island_key
    FROM    transactions
    WHERE   account_id = 101
)
SELECT  account_id,
        txn_type,
        island_key,
        MIN(txn_date)   AS island_start,
        MAX(txn_date)   AS island_end,
        COUNT(*)        AS island_length
FROM    numbered
GROUP BY account_id, txn_type, island_key
ORDER BY island_start
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Session detection — time-gap based grouping

In [ ]:
# Sessions: events are in the same session if gap < 20 minutes
# New session starts when gap >= 20 minutes
print('=== Session detection (20-minute gap threshold) ===')
q = """
WITH gaps AS (
    SELECT  event_id,
            account_id,
            event_time,
            LAG(event_time) OVER (
                PARTITION BY account_id ORDER BY event_time
            )  AS prev_time,
            -- gap in minutes (julianday returns days, x1440 for minutes)
            ROUND((JULIANDAY(event_time) -
                   JULIANDAY(LAG(event_time) OVER (
                       PARTITION BY account_id ORDER BY event_time
                   ))) * 1440, 1) AS gap_minutes
    FROM    login_events
),
session_starts AS (
    SELECT  *,
            CASE WHEN prev_time IS NULL OR gap_minutes >= 20 THEN 1
                 ELSE 0
            END AS is_new_session
    FROM    gaps
)
SELECT  event_id,
        account_id,
        event_time,
        gap_minutes,
        is_new_session,
        SUM(is_new_session) OVER (
            PARTITION BY account_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )  AS session_id
FROM    session_starts
ORDER BY account_id, event_time
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Summarise sessions
print()
print('=== Session summary per account ===')
q2 = """
WITH gaps AS (
    SELECT event_id, account_id, event_time,
           ROUND((JULIANDAY(event_time) - JULIANDAY(LAG(event_time) OVER (
               PARTITION BY account_id ORDER BY event_time))) * 1440, 1) AS gap_minutes
    FROM login_events
),
session_starts AS (
    SELECT *, CASE WHEN gap_minutes IS NULL OR gap_minutes >= 20 THEN 1 ELSE 0 END AS is_new
    FROM gaps
),
with_session AS (
    SELECT *, SUM(is_new) OVER (PARTITION BY account_id ORDER BY event_time
                                ROWS UNBOUNDED PRECEDING) AS session_id
    FROM session_starts
)
SELECT  account_id,
        session_id,
        MIN(event_time)  AS session_start,
        MAX(event_time)  AS session_end,
        COUNT(*)         AS events,
        ROUND((JULIANDAY(MAX(event_time)) - JULIANDAY(MIN(event_time))) * 1440, 1)
                         AS duration_minutes
FROM    with_session
GROUP BY account_id, session_id
ORDER BY account_id, session_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Island key is not unique across partitions**
The island key `rn_all - rn_group` resets per partition. Two different partitions may produce the same island_key value (e.g. both account 101 and account 103 may have island_key = 0). Always include the partition column(s) in GROUP BY when summarising islands: `GROUP BY account_id, txn_type, island_key`.

**2. Date gaps depend on expected frequency**
A gap of 2 days in daily data is a gap. A gap of 2 days in weekly data is not. Always define "gap" explicitly relative to the expected frequency of your data. The LAG-based gap calculation in this notebook detects any interval > 1 day — adjust the threshold for your domain.

**3. The island technique requires a consistent ORDER BY**
`ROW_NUMBER() OVER (ORDER BY txn_date)` without a tiebreaker produces non-deterministic island keys when dates are tied. Add `txn_id` or another unique column as a tiebreaker. Inconsistent order means inconsistent island groupings.

**4. Session detection threshold is a business parameter, not a SQL rule**
The 20-minute gap threshold for sessions is a choice, not a universal rule. Different products use different thresholds (30 minutes is common in web analytics). Make the threshold a clear constant in your query, not buried in a WHERE clause, so it's easy to adjust.

**5. SUM(is_new_session) running total requires ROWS, not RANGE**
`SUM(is_new_session) OVER (ORDER BY event_time ROWS UNBOUNDED PRECEDING)` correctly increments the session counter. Using the default RANGE mode with tied timestamps could assign the same session number to events that should be in different sessions. Always use `ROWS` for session counters.

**6. Gaps in non-integer sequences need domain-specific logic**
Integer gaps (missing IDs 1,2,4,5 → missing 3) are straightforward. Date gaps require knowing the expected interval. For irregular event data (no fixed interval), gaps are relative — define what gap size is meaningful for your use case before writing the detection query.


---
*sql_methods_library - Samantha McGarrigle*